In [2]:
from src.config import RAW_DATA_DIR

from src.data.loader import load_meld
from src.features.tfidf import TFIDFFeatures

from src.models.logistic_regression import LogisticRegressionClassifier
from src.models.svm import LinearSVMClassifier

from src.evaluation.metrics import evaluate
from src.evaluation.results import (
    save_metrics,
    save_metadata,
    save_confusion_matrix,
)

In [3]:
meld = load_meld(RAW_DATA_DIR / "meld")

y_train = meld.train["emotion"]
y_test = meld.test["emotion"]

In [4]:
experiments = [
    {
        "name": "logistic_unigram",
        "model": "logistic",
        "ngram_range": (1, 1),
    },
    {
        "name": "logistic_bigram",
        "model": "logistic",
        "ngram_range": (1, 2),
    },
    {
        "name": "logistic_trigram",
        "model": "logistic",
        "ngram_range": (1, 3),
    },
    {
        "name": "svm_unigram",
        "model": "svm",
        "ngram_range": (1, 1),
    },
    {
        "name": "svm_bigram",
        "model": "svm",
        "ngram_range": (1, 2),
    },
    {
        "name": "svm_trigram",
        "model": "svm",
        "ngram_range": (1, 3),
    },
]

In [5]:
def create_model(model_name):

    if model_name == "logistic":
        return LogisticRegressionClassifier()

    if model_name == "svm":
        return LinearSVMClassifier()

    raise ValueError(f"Unknown model: {model_name}")

In [ ]:
for experiment in experiments:

    print(f"\n===== {experiment['name']} =====")

    features = TFIDFFeatures(
        ngram_range=experiment["ngram_range"],
    )

    X_train = features.fit_transform(
        meld.train["text"]
    )

    X_test = features.transform(
        meld.test["text"]
    )

    model = create_model(
        experiment["model"]
    )

    model.fit(
        X_train,
        y_train,
    )

    predictions = model.predict(
        X_test
    )

    results = evaluate(
        y_test,
        predictions,
    )

    print(
        f"Accuracy : {results.accuracy:.4f}"
    )

    print(
        f"Macro F1 : {results.macro_f1:.4f}"
    )

    save_metrics(
        results,
        experiment["name"],
    )

    save_metadata(
        experiment["name"],
        model=experiment["model"],
        dataset="MELD",
        ngram_range=experiment["ngram_range"],
    )

    save_confusion_matrix(
        results,
        experiment["name"],
    )


===== logistic_unigram =====
Accuracy : 0.5268
Macro F1 : 0.2410

===== logistic_bigram =====
Accuracy : 0.5318
Macro F1 : 0.2470

===== logistic_trigram =====
Accuracy : 0.5276
Macro F1 : 0.2327

===== svm_unigram =====
Accuracy : 0.5057
Macro F1 : 0.2788

===== svm_bigram =====
Accuracy : 0.5084
Macro F1 : 0.2874

===== svm_trigram =====


In [ ]:
from src.evaluation.comparison import compare_experiments

comparison = compare_experiments(
    [
        "logistic_unigram",
        "logistic_bigram",
        "logistic_trigram",
        "svm_unigram",
        "svm_bigram",
        "svm_trigram",
    ]
)

comparison